# Context prediction from scratch (Doersch et al., 2015)

My own implementation of the **relative patch position** pretext task:

1. a 3×3 grid of patches is sampled in an unlabeled STL-10 image (with gaps and random jitter so
   that edges cannot simply be continued across patches);
2. a siamese AlexNet receives the center patch and one of its 8 neighbours and predicts the
   neighbour's position (8-way classification, chance = 12.5%);
3. the shared patch encoder (`fc6`, 4096-d) is then **frozen** and evaluated with a linear
   classifier on CIFAR-10 (or STL-10).

The code lives in [`context_prediction.py`](../context_prediction.py); the same pipeline can be
run from the command line with `python context_prediction.py --epochs 100`.

In [ ]:
import os
import sys

if "google.colab" in sys.modules:
    !git clone -q https://github.com/<your-username>/self-supervised-learning.git /content/self-supervised-learning
    %cd /content/self-supervised-learning
else:
    sys.path.append("..")

import random

import matplotlib.pyplot as plt
import torch
from torchvision import datasets

import context_prediction as cp
from ssl_eval import per_class_accuracy, plot_grouped_bars, plot_history

%matplotlib inline
random.seed(0)
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = "./data"
OUTPUT_DIR = "outputs/context_prediction"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CONFIG = dict(patch_size=24, gap=8, jitter=4, grayscale_prob=0.0)
EPOCHS, LR, BATCH_SIZE = 100, 0.006, 128
PRETEXT_SPLIT = "train"   # "train+unlabeled" uses the 100k unlabeled images (much longer)
PROBE_DATASET = "cifar10"

## 1. Visualising the pretext task

In [ ]:
def show_patch_pairs(dataset, n=6, **patch_kwargs):
    """
    Display sampled (center, neighbour) pairs with their position label.

    Parameters
    ----------
    dataset : torchvision.datasets.STL10
        Dataset returning PIL images.
    n : int, default=6
        Number of pairs.
    **patch_kwargs
        Passed to :func:`context_prediction.sample_patch_pair`.

    Returns
    -------
    None
    """
    fig, axs = plt.subplots(n, 3, figsize=(6, 2.1 * n))
    for row in range(n):
        image, _ = dataset[random.randrange(len(dataset))]
        center, neighbor, label = cp.sample_patch_pair(image, **patch_kwargs)
        for ax, img, title in zip(axs[row], (image, center, neighbor),
                                  ("image", "center", f"neighbour: {cp.POSITION_NAMES[label]}")):
            ax.imshow(img)
            ax.set_title(title, fontsize=9)
            ax.axis("off")
    plt.tight_layout()
    plt.show()


raw_stl = datasets.STL10(DATA_ROOT, split="train", download=True)
patch_geometry = {k: CONFIG[k] for k in ("patch_size", "gap", "jitter")}
show_patch_pairs(raw_stl, **patch_geometry)

## 2. Pretext training

In [ ]:
train_loader = cp.make_pretext_loader(DATA_ROOT, PRETEXT_SPLIT, BATCH_SIZE, shuffle=True, **CONFIG)
test_loader = cp.make_pretext_loader(DATA_ROOT, "test", BATCH_SIZE, shuffle=False, **CONFIG)

model = cp.build_siamese_alexnet().to(device)
ckpt = os.path.join(OUTPUT_DIR, "siamese_alexnet.pth")
pretext_history = cp.train_pretext(model, train_loader, test_loader, device, EPOCHS, LR,
                                   checkpoint_path=ckpt)
plot_history(pretext_history, "Pretext task: relative patch position",
             os.path.join(OUTPUT_DIR, "pretext_history.png"))

## 3. Linear evaluation of the frozen encoder

In [ ]:
model.load_state_dict(torch.load(ckpt, map_location=device))
probe_train, probe_test, classes = cp.make_probe_loaders(PROBE_DATASET, DATA_ROOT)
train_x, train_y = cp.extract_features(model, probe_train, device)
test_x, test_y = cp.extract_features(model, probe_test, device)

probe_history, preds = cp.train_linear_probe(train_x, train_y, test_x, test_y, len(classes), device)
plot_history(probe_history, f"Linear probe on {PROBE_DATASET}",
             os.path.join(OUTPUT_DIR, "probe_history.png"))
plot_grouped_bars({"context prediction": per_class_accuracy(preds, test_y.numpy(), len(classes))},
                  classes, title=f"Per-class accuracy ({PROBE_DATASET})", ylim=(0, 105),
                  save_path=os.path.join(OUTPUT_DIR, "probe_per_class.png"));

## 4. Baseline: random (untrained) encoder

A linear probe on a randomly initialised network tells how much of the accuracy actually comes
from self-supervised pretraining.

In [ ]:
random_model = cp.build_siamese_alexnet().to(device)
rx_train, _ = cp.extract_features(random_model, probe_train, device)
rx_test, _ = cp.extract_features(random_model, probe_test, device)
random_history, _ = cp.train_linear_probe(rx_train, train_y, rx_test, test_y, len(classes), device)

print(f"random encoder : {random_history[-1]['test_acc']:.2f}%")
print(f"context pred.  : {probe_history[-1]['test_acc']:.2f}%")

## Observations

*Fill in after running.* Useful angles: pretext accuracy vs chance (12.5%), gain over the random
encoder, effect of `grayscale_prob` (chromatic aberration, see notebook 02) and of training on
`train+unlabeled`.